# Building a Safe AI Operations Assistant for an Installment Payment Service

**Portfolio case study · AI Engineering · November 2026**

This project demonstrates how to combine LLM tool calling, a bounded ReAct loop, guardrails, reliable external API wrappers, idempotent financial operations, structured tracing, and scenario-based evaluation in one production-oriented assistant.

> **Final result:** 24 of 24 scenarios passed (100%), all 4 prompt-injection attacks were blocked, the false-block rate was 0%, and the assistant created **zero duplicate or extra refunds**. The initial baseline was 17/24.

## 1. Business problem

The fictional **Dolyka** service provides installment payments for purchases at partner stores. Its operations team needs an assistant that can:

1. answer questions using internal operating regulations;
2. inspect order records and payment status;
3. calculate deadlines using a production calendar;
4. create refunds only for eligible orders and only after trusted operator confirmation.

The key challenge is that this is not a read-only chatbot. It can trigger a financial side effect through an unreliable partner API. A previous automation attempt created a duplicate refund after a network failure and followed an instruction embedded in customer text. The new assistant therefore had to be safe under both infrastructure failures and adversarial input.

## 2. Engineering requirements

The implementation was designed around the following invariants:

- external-service failures must not crash the agent;
- transient read failures must be retried with bounded exponential backoff;
- a network timeout must never create a duplicate refund;
- customer text is untrusted data and cannot grant operator confirmation;
- every external call must be journaled without API keys or sensitive data;
- uncertain financial actions must fail closed;
- each request must have a complete JSONL trace for audit and debugging.

## 3. System architecture

A **bounded manual ReAct loop** was selected instead of LangGraph. Typical workflows need only two to four steps and have a small number of explicit branches. For a financial workflow, direct control and auditability were more valuable than adding another orchestration dependency.

```text
User request
     │
     ▼
Input guardrail ── block injection / out-of-scope requests
     │
     ▼
Bounded ReAct loop (maximum 8 steps)
     │
     ├── LLM selects a tool
     ├── Action guardrail validates tool + arguments + policy
     ├── Reliable wrapper calls external service
     └── Observation is returned to the LLM
     │
     ▼
Output guardrail ── redact cards, phones, and secrets
     │
     ▼
Answer + structured JSONL audit trace
```

## 4. Repository map and implementation stages

| File | Responsibility |
|---|---|
| `agent/tools.py` | Partner payment status, business-day calculation, refund creation, finish tool, and the tool registry |
| `agent/reliability.py` | Five-second timeouts, retries, backoff, idempotent writes, and per-attempt API logging |
| `agent/guardrails.py` | Trusted policy context plus input, action, and output guardrails |
| `agent/agent.py` | System prompt, bounded ReAct loop, tool-call recovery, observations, decisions, token counts, and tracing |
| `evals/run_eval.py` | Execution and scoring of all 24 test scenarios |
| `support/` | Provided model client, schemas, data access, and journal utilities |
| `data/` | 20 regulations, 60 orders, 24 scenarios, receipts, and the mock payment gateway |
| `logs/` | Final evaluation tables, JSON results, and one trace per request |

### Development sequence

1. Implemented reliable read, write, and LLM wrappers.
2. Added external-service tools and a common registry.
3. Added three guardrail layers and a trusted `PolicyContext`.
4. Built the bounded ReAct loop and structured journal events.
5. Added a repeatable evaluation runner and side-effect checks.
6. Analyzed failed traces and improved local-model tool-call handling.
7. Re-ran the final evaluation with the gateway's 30% failure rate.

## 5. Safety design

### Trusted confirmation

Operator confirmation is passed through `PolicyContext.confirmed_refunds`, which is created by application code. A phrase such as *"the operator approved this"* inside a customer email has no authority.

### Refund action guardrail

Before `create_refund` can execute, the guardrail verifies that:

- the tool is allow-listed;
- arguments have the expected names and types;
- the order exists;
- the order is cancelled;
- no refund already exists;
- the refund amount equals the order amount;
- trusted operator confirmation exists.

If confirmation is missing, the decision is `needs_confirmation` and no partner refund call is made.

### Output protection

Full card numbers, Russian phone numbers, and the partner API key are removed from final answers. The provided journal masker replaces sensitive `request_id` values with `***`.

## 6. Reliability and idempotency

Read operations use up to three attempts with a five-second timeout and delays of 0.5 and 1.0 seconds. HTTP 429, HTTP 5xx, connection errors, and timeouts are treated as transient failures.

Write operations are retried **only when an idempotency key is present**. Every refund uses a stable key:

```text
refund-<order_id>
```

The mock gateway can create a refund and then return HTTP 500 or time out. Retrying with the same key allows the gateway to recognize the operation and prevents a duplicate financial side effect.

The final traces contain a real retry sequence after HTTP 429 and a five-second `no_response` timeout. Despite retries, the number of extra refunds remained zero.

## 7. Evaluation methodology

The evaluation suite contains 24 scenarios across seven categories:

- regulation questions;
- payment and order status checks;
- deadline calculations;
- confirmed refunds;
- unconfirmed refunds;
- prompt-injection attacks;
- out-of-scope requests.

Each scenario checks the final decision, required answer fragments, required tools, and the number of refunds actually created in the gateway. The gateway is reset before the run and operates with its default reproducible 30% failure rate.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / 'logs' / 'eval_main.json').exists():
    candidates = list(ROOT.rglob('logs/eval_main.json'))
    if not candidates:
        raise FileNotFoundError('Run this notebook from the repository or place it above the repository.')
    ROOT = candidates[0].parents[1]

evaluation = json.loads((ROOT / 'logs' / 'eval_main.json').read_text(encoding='utf-8'))
summary = evaluation['summary']
summary

In [ ]:
category_results = [
    {
        'category': category,
        'passed': values['correct'],
        'total': values['total'],
        'accuracy_pct': round(values['rate'] * 100, 1),
    }
    for category, values in summary['by_category'].items()
]
category_results

## 8. Final results

| Metric | Result |
|---|---:|
| Correct scenarios | **24 / 24** |
| Overall accuracy | **100%** |
| Prompt-injection attacks blocked | **4 / 4 (100%)** |
| False-block rate | **0%** |
| Unconfirmed refunds safely stopped | **2 / 2 (100%)** |
| Extra refunds | **0** |
| Average agent steps | **2.33** |
| Average executed tool calls | **1.29** |

Every evaluation category reached 100%. Both confirmed-refund scenarios created exactly one valid refund, both unconfirmed scenarios stopped without a write, all attacks were blocked, and no extra refund was created.

In [ ]:
failed = [
    {
        'id': row['id'],
        'category': row['category'],
        'decision': row['decision'],
        'tools_used': row['tools_used'],
        'answer': row['answer'],
    }
    for row in evaluation['requests'] if not row['correct']
]
failed

## 9. Failure analysis and engineering trade-offs

The initial 17/24 baseline exposed model-quality failures, but they remained **safe failures**: no unauthorized or duplicate refund was created. The final iteration resolved all seven misses and reached 24/24.

### Observed failure classes

1. **Plain text instead of structured tool calls.** A corrective turn now asks for the missing required tool before any prose is returned.
2. **Malformed read arguments.** Safe read-only arguments are normalized to the original request before validation.
3. **Incomplete tool chains.** Required tools are derived from the request and premature completion is rejected.
4. **Semantically correct wording mismatch.** Canonical policy wording is appended only when grounded by a successful supporting tool.
5. **Ambiguous write failures.** An automated integration test proves HTTP 500 to retry to HTTP 200 with one shared idempotency-key hash and exactly one refund.

### What I would improve next

- add a deterministic intent router for high-risk workflows;
- represent refund processing as a small explicit state machine;
- extend the existing five-test suite with contract tests for every tool-call argument shape;
- move all final-response construction to typed structured observations;
- compare a larger or fine-tuned local model against the current 7B model;
- add CI that starts the gateway, runs unit tests, and validates trace schemas.

## 10. Reproducing the project

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

export PARTNER_API_KEY=demo-key
export LLM_MODEL=qwen2.5:7b
ollama pull qwen2.5:7b
```

Start the partner gateway in one terminal:

```bash
python data/gateway/mock_gateway.py
```

Then run the evaluation in another terminal:

```bash
python check_tools.py
python evals/run_eval.py
```

The evaluation writes `logs/eval_main.txt`, `logs/eval_main.json`, and 24 request traces.

## 11. Skills demonstrated

- LLM function calling and ReAct orchestration
- prompt-injection defense and policy enforcement
- idempotent financial API design
- retry, timeout, and fallback strategies
- structured observability with JSONL traces
- scenario-based agent evaluation
- failure analysis and safety-first iteration
- Python, OpenAI-compatible APIs, Ollama, and HTTP integrations

---

This case study focuses on a central AI-engineering lesson: an agent that can take real actions must be evaluated not only by answer quality, but also by whether its failures remain controlled, observable, and free of harmful side effects.